# Session 2: From one neuron to a network

**259.075 Data-integrated Algorithmic Design Processes II** · Session 2 of 7, about 75 minutes

Last week your line used one input, compactness, and stopped at an error of about 50. The U-value
of the wall sits in the data but not in the model. Today you hand it over, by foot on four
buildings. The error collapses, and right afterwards we take the result apart, because it is worth
much less than it looks.

The second part matters more. A straight line, however many inputs you give it, stays straight,
and most things worth predicting are not. You will see what goes wrong, watch the obvious repair
fail, and then build the smallest thing that works: a few kinks, added up.

Before most cells you are asked to **guess first**: say or write down what you expect, then run
the cell and compare.

Timing: Part 1 (two inputs) about 40 minutes, Part 2 (the kink) about 30, tests 5. The three parts
marked *optional* after Part 2 are for whoever is fast. The homework uses the last of them; it runs
whether or not you read it.

## Part 1: two inputs

### Step 1: one building, two inputs

Four real buildings from our table of Vienna: compactness, U-value of the wall and heating demand.

<div style="border-left: 5px solid #d8a24a; background-color: #fdf6e3; color: #222; padding: 10px 16px; margin: 6px 0;">

**💡 About the data**

The heating demand in this table is **not measured**. It was computed from the U-value and the
compactness by a fixed rule. Keep that in mind: a model that hits it exactly has found the rule
again, not learned anything about houses.

</div>

With two inputs each one gets its own weight: predicted demand = w1 × compactness + w2 × U-value
+ b. This is a **weighted sum**: each input times its weight, added up, plus the bias. Each input
column is also called a **feature**; choosing them is a design decision.

One line in the cell is yours. Right above it stands last week's model with one input, `w * c + b`;
yours has one more term. **Guess first:** with the guess w1 = 10, w2 = 40 and b = 30, what does the
first building get?

In [ ]:
# The four buildings, in the same order in all three lists. Building 0
# is from 1860-1883, 1 from 1884-1918, 2 from 1946-1976, 3 from after
# 1976.
compactness = [6.263, 4.236, 5.195, 5.175]
u_value = [1.50, 1.45, 1.30, 0.70]
# Heating demand in kWh/m²a, computed, not measured
demand = [167.36, 146.03, 145.36, 107.98]


# Last week's model with one input: weight w times compactness c,
# plus the bias b. return hands the result back. It is only here as
# the example for your line below.
def predict_one(c, w, b):
    return w * c + b


# The model with two inputs: c is the compactness, u the U-value
def predict(c, u, w1, w2, b):
    # Hand back: w1 times c, plus w2 times u, plus b
    return w1 * c + w2 * u + b


# A first guess for the two weights and the bias
w1 = 10
w2 = 40
b = 30

# Building 0, piece by piece. compactness[0] is the first value of
# the list: Python counts from 0. Each input times its weight:
part_1 = w1 * compactness[0]
part_2 = w2 * u_value[0]
# Added up, plus the bias
total_0 = part_1 + part_2 + b
# :.2f in the curly brackets prints a number with two decimals
print(f"building 0: {part_1:.2f} + {part_2:.2f} + {b} = {total_0:.2f}")

# The same with your function, for all four buildings, collected in
# a list that starts empty
predictions = []
# range(4) gives the positions 0, 1, 2, 3: one per building
for n in range(4):
    # The prediction for building n
    p = predict(compactness[n], u_value[n], w1, w2, b)
    # append adds the value at the end of the list
    predictions.append(p)
    # Show it next to the value in the table
    print(f"building {n}: predicted {p:.2f}, table says {demand[n]}")

<details>
<summary>Answer</summary>

    return w1 * c + w2 * u + b

Building 0: 10 × 6.263 + 40 × 1.5 + 30 = 62.63 + 60 + 30 = 152.63, against 167.36 in the table.
Building 3, the one from after 1976 with the good wall (U-value 0.7), gets 109.75 against 107.98:
the only one the guess predicts too high.

</details>

### Step 2: how wrong, as one number

The same MSE (mean squared error) as last week: each miss squared, a running total, divided by how
many. **Guess first:** which building adds most to the total?

In [ ]:
# The running total of squared misses, starting at 0
total = 0
for n in range(4):
    # The miss: predicted (from step 1) minus actual
    miss = predictions[n] - demand[n]
    # Squared: always positive, big misses count more
    square = miss * miss
    # Add it to the running total
    total = total + square
    # Show all three numbers for this building
    print(f"building {n}: miss {miss:7.2f}, squared {square:7.2f}, "
          f"total so far {total:7.2f}")

# Divide by how many buildings: the mean
guess_mse = total / 4
# Show the division
print(f"MSE = {total:.2f} / 4 = {guess_mse:.2f}")

<details>
<summary>Answer</summary>

Building 1: it misses by −15.67, squared 245.55. Building 0 is close behind with 216.97. The four
squares add up to 595.84, divided by 4 the MSE is 148.96.

</details>

We need this for many values of w1, w2 and b, and later also for rescaled inputs, so the next cell
packs the same loop into a function that also takes the two input lists. Nothing new in it.

In [ ]:
def mse(w1, w2, b, inputs_1, inputs_2):
    # inputs_1 and inputs_2: the two input lists, one value per
    # building. The demand comes from the list of step 1.
    total = 0
    for n in range(4):
        # Prediction, miss, square, running total, as above
        p = predict(inputs_1[n], inputs_2[n], w1, w2, b)
        miss = p - demand[n]
        square = miss * miss
        total = total + square
    # Hand back the mean
    return total / 4


# Check: the same number as the loop above
check = mse(10, 40, 30, compactness, u_value)
# Show it
print(f"MSE of the guess: {check:.2f}")

### Step 3: three knobs, three gradients

Now there are three knobs, w1, w2 and b. For each the same trick as last week: turn that one knob a
tiny step, 0.001, leave the other two, and see how the MSE changes. The change divided by 0.001 is
the **gradient** for that knob. First by foot for w1. **Guess first:** will the gradient for w1 be
positive or negative? (Step 1 tells you: were the predictions mostly too high or too low?)

In [ ]:
# The size of the tiny step
tiny = 0.001
# The MSE where we stand
here = mse(10, 40, 30, compactness, u_value)
# The MSE with w1 a tiny step bigger, w2 and b unchanged
further = mse(10 + tiny, 40, 30, compactness, u_value)
# How much the MSE changed
change = further - here
# Change per unit of w1: the gradient
gradient_w1 = change / tiny

# Show each step of the arithmetic
print(f"MSE here:            {here:.6f}")
print(f"MSE with w1 + 0.001: {further:.6f}")
print(f"change:              {change:.6f}")
print(f"change / 0.001:      {gradient_w1:.2f}")

<details>
<summary>Answer</summary>

Negative, about −104.35: three of the four predictions were too low, so making w1 bigger lifts them
towards the table, and the MSE falls by 0.104 for the tiny step.

</details>

The same for all three knobs, packed into one function. **Guess first:** which of the three
gradients will be largest in size, ignoring the sign?

In [ ]:
def gradients(w1, w2, b, inputs_1, inputs_2):
    # The size of the tiny step
    tiny = 0.001
    # The MSE where we stand
    here = mse(w1, w2, b, inputs_1, inputs_2)
    # One knob a tiny step further at a time, the others unchanged
    further_w1 = mse(w1 + tiny, w2, b, inputs_1, inputs_2)
    further_w2 = mse(w1, w2 + tiny, b, inputs_1, inputs_2)
    further_b = mse(w1, w2, b + tiny, inputs_1, inputs_2)
    # Change per unit, for each knob
    g_w1 = (further_w1 - here) / tiny
    g_w2 = (further_w2 - here) / tiny
    g_b = (further_b - here) / tiny
    # Hand back all three in one list, in this order
    return [g_w1, g_w2, g_b]


# The three gradients at the guess
g = gradients(10, 40, 30, compactness, u_value)
# Take the three values out of the list by position and show them
print(f"gradient for w1: {g[0]:8.2f}")
print(f"gradient for w2: {g[1]:8.2f}")
print(f"gradient for b:  {g[2]:8.2f}")

<details>
<summary>Answer</summary>

w1, with −104.35; w2 gets −29.20 and b −20.02. All three are negative, so all three knobs should go
up. Here w1 has the largest gradient because compactness is the largest input, 4 to 6: turning w1
moves every prediction by four to six times as much as turning b.

</details>

One step of the learning loop, as last week: each knob minus learning rate times its gradient. All
three gradients are computed **before** any knob moves. **Guess first:** after one step, is the MSE
lower than 148.96?

In [ ]:
# Start from the guess
w1 = 10
w2 = 40
b = 30
# How big the step is
learning_rate = 0.015

# All three gradients first, where we stand
g = gradients(w1, w2, b, compactness, u_value)
# Then the three updates: minus, so each goes against its gradient
new_w1 = w1 - learning_rate * g[0]
new_w2 = w2 - learning_rate * g[1]
new_b = b - learning_rate * g[2]
# Show the arithmetic of each update
print(f"w1 = {w1} - {learning_rate} * ({g[0]:.2f}) = {new_w1:.3f}")
print(f"w2 = {w2} - {learning_rate} * ({g[1]:.2f}) = {new_w2:.3f}")
print(f"b  = {b} - {learning_rate} * ({g[2]:.2f}) = {new_b:.3f}")

# The MSE of the line after the step
step_mse = mse(new_w1, new_w2, new_b, compactness, u_value)
# Show it
print(f"MSE after one step: {step_mse:.2f}")

<details>
<summary>Answer</summary>

Much lower: 49.25. w1 goes from 10 to 11.565, w2 from 40 to 40.438, b from 30 to 30.300.

</details>

### Step 4: the loop crawls

Now the loop, three thousand steps at the same learning rate. To compare fairly with what comes
later, it starts from a blank model: both weights 0, the bias at the average demand, 141.68, so
every building first gets the average. **Guess first:** one step took the MSE from 148.96 to 49.25.
After 3000 steps, will it be close to 0?

In [ ]:
def learn(inputs_1, inputs_2, learning_rate, steps, print_every):
    # Blank start: both weights 0, the bias at the average demand.
    # sum(...) adds up all values of a list.
    w1 = 0
    w2 = 0
    b = sum(demand) / 4
    # range(1, steps + 1) counts 1, 2, ... steps (the end is left out)
    for step in range(1, steps + 1):
        # All three gradients first ...
        g = gradients(w1, w2, b, inputs_1, inputs_2)
        # ... then the three updates
        w1 = w1 - learning_rate * g[0]
        w2 = w2 - learning_rate * g[1]
        b = b - learning_rate * g[2]
        # step % print_every is what is left when step is divided by
        # print_every; it is 0 every print_every steps
        if step % print_every == 0:
            # The MSE now, and a line with all numbers
            now = mse(w1, w2, b, inputs_1, inputs_2)
            print(f"step {step:4}: w1 = {w1:6.2f}, w2 = {w2:6.2f}, "
                  f"b = {b:6.2f}, MSE = {now:.3f}")
    # Hand back the three numbers in one list
    return [w1, w2, b]


# 3000 steps on the inputs as they are, a line every 500 steps
raw_result = learn(compactness, u_value, 0.015, 3000, 500)

<details>
<summary>Answer</summary>

No: the MSE is still at 18.03 after 3000 steps, and still falling slowly. The bias sits at 52.01.
w1 even went negative first, to −4.43 at step 500, and is now creeping back up.

</details>

Nothing is broken. The trouble is that compactness is about 5 for every building. Raising w1 by one
lifts every prediction by about 5, much as raising b by 5 would: two knobs doing nearly the same
thing. The loop cannot tell them apart and slides along a long, narrow valley between them. A
bigger learning rate does not help: change 0.015 to 0.04 in the last line of the cell and the
numbers blow up instead.

> The negative gradient points downhill, not towards the lowest point. In a narrow valley those
> are two very different directions.

### Step 5: rescaling by foot

The repair is to measure every input in the same currency. For each input column: subtract its
average, then divide by its **standard deviation**, the usual distance of a value from the average.
By foot, on three compactness values first. **Guess first:** after subtracting the average, what
will the three values add up to?

In [ ]:
values = [6.263, 4.236, 5.195]

# The average: add up, divide by how many
average = sum(values) / 3
# Show it
print(f"average: {average:.4f}")

# The standard deviation, step by step: each value's distance from
# the average, squared, added up ...
total = 0
for v in values:
    distance = v - average
    square = distance * distance
    total = total + square
    # Show distance and square for this value
    print(f"  {v}: distance {distance:7.4f}, squared {square:.4f}")
# ... divided by how many: the mean of the squares ...
mean_square = total / 3
# ... and its square root. ** means "to the power"; ** 0.5 is the
# square root, which turns the squared distances back into distances
spread = mean_square ** 0.5
# Show it
print(f"standard deviation: {spread:.4f}")

# Rescaled: distance from the average, divided by the spread
for v in values:
    rescaled = (v - average) / spread
    # Show it
    print(f"  {v} rescaled: {rescaled:7.4f}")

<details>
<summary>Answer</summary>

0, up to rounding: 1.0317 − 0.9953 − 0.0363 = 0.0001, and exactly 0 without the rounding. That is
what an average is. The spread is 0.8279; the first value lies 1.0317 above the average, which is
1.2461 spreads. Rescaled values say "how many spreads above or below the average", so they always
average 0 and spread 1.

</details>

Rescaling this way is also called **normalisation**. The same as two functions, for both input
columns of our four buildings.

In [ ]:
def average_and_spread(values):
    # How many values there are: len(...) counts them
    count = len(values)
    # The average
    average = sum(values) / count
    # The spread, as in the cell above
    total = 0
    for v in values:
        distance = v - average
        total = total + distance * distance
    spread = (total / count) ** 0.5
    # Hand back two values, in this order: average, then spread
    return average, spread


def rescale(values, average, spread):
    # A new list with each value rescaled
    result = []
    for v in values:
        # Distance from the average, in spreads
        rescaled = (v - average) / spread
        result.append(rescaled)
    return result


# Catch the two values in two names, in the same order
c_average, c_spread = average_and_spread(compactness)
u_average, u_spread = average_and_spread(u_value)
# Show them
print(f"compactness: average {c_average:.4f}, spread {c_spread:.4f}")
print(f"U-value:     average {u_average:.4f}, spread {u_spread:.4f}")

# The two rescaled input lists
c_rescaled = rescale(compactness, c_average, c_spread)
u_rescaled = rescale(u_value, u_average, u_spread)
# Show them, one building per line
for n in range(4):
    print(f"building {n}: {c_rescaled[n]:6.3f}  {u_rescaled[n]:6.3f}")

### Step 6: the loop on rescaled inputs, and the weights in real units

The same loop, the same learning rate and the same start, on the rescaled inputs, and only 400
steps instead of 3000. **Guess first:** where will the MSE be at step 400?

In [ ]:
# 400 steps on the rescaled inputs, a line every 100 steps
scaled_result = learn(c_rescaled, u_rescaled, 0.015, 400, 100)

<details>
<summary>Answer</summary>

At 0.000 already by step 300. The average of each rescaled column is 0, so w1 and w2 no longer move
every prediction up and down together with b; each knob does its own job.

</details>

The price: the weights now mean "demand per spread of the input", which nobody can check against a
building code. Converting them back is arithmetic. The model computes w × (c − average) / spread,
and that is the same as (w / spread) × c − w × average / spread. So the weight per real unit is
w / spread, and the pieces w × average / spread go into the bias.

In [ ]:
# The three learned numbers, taken out of the list by position
w1_s = scaled_result[0]
w2_s = scaled_result[1]
b_s = scaled_result[2]

# Weight per real unit: the weight divided by the spread of its input
w1_real = w1_s / c_spread
w2_real = w2_s / u_spread
# The bias takes up what is left: w * average / spread, for each input
shift_1 = w1_s * c_average / c_spread
shift_2 = w2_s * u_average / u_spread
b_real = b_s - shift_1 - shift_2

# Show the arithmetic
print(f"per unit of compactness: {w1_s:.3f} / {c_spread:.4f} = {w1_real:.3f}")
print(f"per W/m²K of U-value:    {w2_s:.3f} / {u_spread:.4f} = {w2_real:.3f}")
print(f"bias: {b_s:.3f} - {shift_1:.3f} - {shift_2:.3f} = {b_real:.3f}")

Nine, sixty-two, eighteen, nearly. Those are the numbers of the rule that computed the heating-demand
column in the first place:

    heating demand = 18 + 62 × U-value + 9 × compactness

The last digits are off a little, partly because the table stores rounded numbers, partly because
the loop with its tiny step only gets close. The model discovered nothing. It fished back out a
calculation rule somebody had put in. The rule carries no measurement noise, which is why the
recovery is this clean, and the U-values are textbook values per construction period, not
measurements. Against measured consumption none of this would look like it does here.

> A model that reproduces its own generating formula has learned arithmetic, not buildings.

What is worth keeping from this part is the rescaling. You need it for the rest of the term.

### Step 7: the shortcuts

**numpy** gives you lists you can compute with in one go: an **array**. Three kinds of shortcut,
each next to the loop it replaces:

- **element by element**: `*`, `+`, `-`, `**` on an array work on every element separately;
- **weighted sums**: `@` multiplies each row of a table with the weights and adds up;
- **summaries**: `np.mean(...)`, or `.mean(axis=0)` for each column of a table.

**Guess first:** what is `[1, 2, 3] * 2` for a plain list?

In [ ]:
# "import numpy as np": load numpy and call it np for short
import numpy as np

# A plain list times 2: the list is REPEATED
doubled_list = [1, 2, 3] * 2
print("list * 2: ", doubled_list)
# An array times 2: every ELEMENT is multiplied
doubled_array = np.array([1, 2, 3]) * 2
print("array * 2:", doubled_array)

# A table with one row per building: [compactness, U-value]. First a
# list of rows ...
rows = []
for n in range(4):
    row = [compactness[n], u_value[n]]
    rows.append(row)
# ... then np.array turns the list of rows into a table
X4 = np.array(rows)
d4 = np.array(demand)
print(X4)

# Weighted sum: @ does "weight times input, added up" for every row
weights = np.array([10, 40])
p4 = X4 @ weights + 30
print("predictions with @:   ", p4)
print("predictions by foot:  ", predictions)

# Element by element: all misses, then all squares
misses = p4 - d4
squares = misses ** 2
# A summary: the mean of all squares
mse_numpy = np.mean(squares)
print(f"MSE with numpy: {mse_numpy:.2f}, by foot: {guess_mse:.2f}")

# Summaries per column: axis=0 means "go down the columns";
# .std(axis=0) is the standard deviation of each column
averages_4 = X4.mean(axis=0)
spreads_4 = X4.std(axis=0)
print("averages with numpy:", averages_4)
print("by foot:            ", c_average, u_average)
print("spreads with numpy: ", spreads_4)
print("by foot:            ", c_spread, u_spread)

<details>
<summary>Answer</summary>

`[1, 2, 3, 1, 2, 3]`: on a plain list `*` repeats it. On an array it multiplies every element, and
that is why numpy code turns lists into arrays first. Everything else agrees with the loops: the
four predictions, the MSE of 148.96, the averages and the spreads.

</details>

The whole of Part 1 on all 1073 buildings, written with these shortcuts, is in *Optional A* below.

## Part 2: where a plane is not enough

### Step 8: a curve, and the best straight line

With two inputs the model is a plane, and everything so far fitted because the target was made to
lie on one. The next target does not. The compactness values are the real ones, the target is
invented by this rule:

    target = 40 + 9 × compactness + 45 × sin(0.8 × compactness)

No physics is claimed. The sine, `sin`, is a wave that swings between −1 and 1, so the last term
adds a wave going 45 up and 45 down. Two values to go on: sin(3.2) is nearly 0, sin(4.8) nearly −1.
**Guess first:** is the target at compactness 6 higher or lower than at 4?

In [ ]:
# math has the sine for single numbers
import math

for c in [4.0, 6.0, 8.0]:
    # The straight part of the rule
    straight_part = 40 + 9 * c
    # The angle for the sine, in radians (the unit math.sin expects)
    angle = 0.8 * c
    # The wave
    wave = 45 * math.sin(angle)
    # Both together
    target = straight_part + wave
    print(f"compactness {c}: {straight_part} + {wave:7.2f} = {target:.2f}")

# The same with numpy, element by element: np.sin of a whole array
three = np.array([4.0, 6.0, 8.0])
targets = 40 + 9 * three + 45 * np.sin(0.8 * three)
print("with numpy:", targets)

<details>
<summary>Answer</summary>

Lower: 73.37 at 4 and 49.17 at 6, then 117.24 at 8. Between 4 and 6 the wave pulls the target down
by 42 while the straight part pushes it up by only 18.

</details>

Now for all buildings: the target, and the best straight line through it. **pandas** reads the
table, and `np.polyfit` finds the line in one go, by least squares, the lowest MSE a line can have.
Then what the line leaves over, the **residuals**: actual minus line. **Guess first:** if a line had
the right shape, the residuals would scatter around 0 without a pattern. What will they look like
here?

In [ ]:
# "import pandas as pd": load pandas and call it pd for short
import pandas as pd

# The full table. "../" means one folder up, then into data.
df = pd.read_csv("../data/buildings_vienna.csv")
# One column of the table, as a numpy array
all_compactness = df["compactness"].to_numpy()
# Sorted from small to large, which makes the plot below easier
curve_x = np.sort(all_compactness)
# The target, element by element
curve_y = 40 + 9 * curve_x + 45 * np.sin(0.8 * curve_x)

# np.polyfit(x, y, 1): the best line (degree 1). It hands back an
# array of two numbers: the slope, then the value at x = 0.
line = np.polyfit(curve_x, curve_y, 1)
slope = line[0]
intercept = line[1]
# The line at every building, and what it leaves over
line_y = slope * curve_x + intercept
residuals = curve_y - line_y
# The MSE of the line
line_mse = np.mean(residuals ** 2)
print(f"best line: {slope:.2f} * compactness + {intercept:.2f}")
print(f"its MSE: {line_mse:.1f}")

In [ ]:
# Plotting tools
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Data with the best line on top, residuals underneath
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.65, 0.35],
                    subplot_titles=("the best straight line there is",
                                    "what it leaves over"))
fig.add_trace(go.Scatter(x=curve_x, y=curve_y, mode="markers", name="data",
                         marker=dict(size=4, color="#4c78a8")), row=1, col=1)
fig.add_trace(go.Scatter(x=curve_x, y=line_y, mode="lines", name="line",
                         line=dict(width=4, color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=curve_x, y=residuals, mode="markers",
                         name="residual",
                         marker=dict(size=4, color="#8b5cf6")), row=2, col=1)
fig.add_hline(y=0, row=2, col=1, line_width=1)
fig.update_layout(height=600, xaxis2_title="compactness")
fig.show()

<details>
<summary>Answer</summary>

They swing: whole stretches of buildings sit above the line, whole stretches below, in a wave. No
learning rate and no patience fixes that, because a line has no shape to give. The MSE of the best
line stays at 379.1.

</details>

> When the residuals show a pattern, the model is missing a form, not a better fit.

### Step 9: two lines in a row are one line

The obvious repair is to stack: feed the result of one line into another. First line h = a × x + p,
second line y = k × h + q. By foot with a = 2, p = 1, k = 3, q = −4. **Guess first:** by how much
does y grow each time x grows by 1?

In [ ]:
# The numbers of the two lines
a = 2
p = 1
k = 3
q = -4

for x in [0, 1, 2, 3]:
    # The first line
    h = a * x + p
    # The second line, fed with the result of the first
    out = k * h + q
    # Show both
    print(f"x = {x}: h = {a}*{x} + {p} = {h},  y = {k}*{h} + {q} = {out}")

<details>
<summary>Answer</summary>

By 6 every time: −1, 5, 11, 17. That is k × a = 3 × 2. Writing the two lines into one shows why:

    y = k × (a × x + p) + q = (k × a) × x + (k × p + q) = 6 × x − 1

A number times a number is a number. Whatever a, p, k and q are, two lines in a row are one line
with slope k × a. Stacking lines buys you nothing; something else has to go in between. The slider
below lets you try 25 combinations of a and k.

</details>

In [ ]:
# Slider over 25 combinations of a and k, with p = 1 and q = 2 fixed
xs = np.linspace(0, 10, 100)
pairs = []
for a_value in [-1.5, -0.5, 0.5, 1.5, 2.5]:
    for k_value in [-1.5, -0.5, 0.5, 1.5, 2.5]:
        pairs.append((a_value, k_value))

# One frame per pair, and one slider position per frame
frames = []
steps = []
for a_value, k_value in pairs:
    ys = k_value * (a_value * xs + 1.0) + 2.0
    name = f"{a_value},{k_value}"
    title = f"a = {a_value}, k = {k_value}: still a straight line"
    frames.append(go.Frame(name=name,
                           data=[go.Scatter(x=xs, y=ys, mode="lines")],
                           layout=go.Layout(title=title)))
    steps.append(dict(method="animate", label=name,
                      args=[[name], dict(mode="immediate",
                                         frame=dict(duration=0,
                                                    redraw=True))]))

# The figure starts with the first pair
first = pairs[0]
first_y = first[1] * (first[0] * xs + 1.0) + 2.0
fig = go.Figure(data=[go.Scatter(x=xs, y=first_y, mode="lines",
                                 line=dict(width=4, color="crimson"))],
                frames=frames)
fig.update_layout(height=420, xaxis_title="x", yaxis_title="y",
                  yaxis_range=[-45, 75],
                  title=f"a = {first[0]}, k = {first[1]}: still straight",
                  sliders=[dict(active=0, currentvalue=dict(prefix="a, k = "),
                                steps=steps)])
fig.show()

### Step 10: the kink

What goes in between is almost insultingly simple: every negative number becomes 0, everything else
stays. It is called a *rectified linear unit*, **ReLU** for short, and in Python it is an `if`.
Because it sits between two lines and stops the result from being a line, it is called an
**activation function**.

A neuron with a ReLU is a **ramp**: flat at 0 up to its kink, then climbing. Below, two ramps, one
with its kink at compactness 4 and one at 6, added up with the weights 10 and −25. **Guess first:**
what does the sum give at compactness 8?

In [ ]:
# ReLU: keep a number if it is positive, otherwise 0
def relu(z):
    # Below zero: hand back 0, and the function stops here
    if z < 0:
        return 0.0
    # Only reached when z is not below zero: hand back z itself
    return z


for c in [3.0, 4.0, 5.0, 6.0, 7.0, 8.0]:
    # Ramp 1 has its kink at 4, ramp 2 at 6
    ramp_1 = relu(c - 4)
    ramp_2 = relu(c - 6)
    # Each ramp times its weight, added up
    together = 10 * ramp_1 - 25 * ramp_2
    # Show the arithmetic
    print(f"compactness {c}: ramps {ramp_1}, {ramp_2}, "
          f"10 * {ramp_1} - 25 * {ramp_2} = {together}")

<details>
<summary>Answer</summary>

10 × 4 − 25 × 2 = −10. Up to 4 the sum is 0, from 4 to 6 it climbs by 10 per unit to 20, and from 6
on it falls by 25 − 10 = 15 per unit: 5 at 7, −10 at 8. Two ramps already make a line that goes up
and then comes down again, which no single line can do.

</details>

The numpy shortcut for the ramp is `np.maximum(0, ...)`: element by element, the larger of 0 and each
value. Here it is next to the loop, and drawn.

In [ ]:
# The same six compactness values, as an array
c_values = np.array([3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
# Element by element: c_values - 4, then the larger of 0 and each value
ramp_1_np = np.maximum(0, c_values - 4)
ramp_2_np = np.maximum(0, c_values - 6)
# Element by element again: weights times ramps, added up
together_np = 10 * ramp_1_np - 25 * ramp_2_np
print("sum with numpy:", together_np)

In [ ]:
# The two weighted ramps and their sum, on a fine grid
grid = np.linspace(3, 8, 101)
r1 = 10 * np.maximum(0, grid - 4)
r2 = -25 * np.maximum(0, grid - 6)
fig = go.Figure()
fig.add_trace(go.Scatter(x=grid, y=r1, name="10 × ramp with kink at 4",
                         line=dict(color="#d8a24a", dash="dash")))
fig.add_trace(go.Scatter(x=grid, y=r2, name="−25 × ramp with kink at 6",
                         line=dict(color="#8b5cf6", dash="dash")))
fig.add_trace(go.Scatter(x=grid, y=r1 + r2, name="sum",
                         line=dict(color="crimson", width=4)))
fig.update_layout(height=380, xaxis_title="compactness",
                  title="two ramps, added up")
fig.show()

### Step 11: many ramps make a network

Several ramps side by side, each with its own kink, added up with their own weights, plus a
constant: that is a network with one **layer** of neurons and an output. Neurons in a layer sit next
to each other, not behind each other; the curve is their sum. Here every ramp faces right, flat to
the left of its kink.

Where to put the kinks? The first at the smallest compactness, so one neuron is exactly a straight
line. Then by halving: the middle, then the quarters, and so on, so going from four neurons to five
keeps the four kinks and adds one in between. The collapsed cell below does that; its output shows
the first five.

In [ ]:
# Kinks by repeated halving between low and high, count of them
def kink_positions(low, high, count):
    # The first kink at the low end
    kinks = [low]
    # level 1 halves the range once, level 2 cuts it in quarters, ...
    level = 1
    # while repeats the indented block as long as the condition holds
    while len(kinks) < count:
        # 2 ** level: 2, 4, 8, ... pieces at this level
        pieces = 2 ** level
        # range(1, pieces, 2) gives 1, 3, 5, ...: only the new cuts
        for i in range(1, pieces, 2):
            if len(kinks) < count:
                position = low + (high - low) * i / pieces
                kinks.append(position)
        level = level + 1
    return kinks


# .min() and .max(): the smallest and largest value of an array
lowest = curve_x.min()
highest = curve_x.max()
five = kink_positions(lowest, highest, 5)
for kink in five:
    print(f"kink at {kink:.3f}")

The output weights: the output is only a weighted sum of the ramps, so the best weights can be
computed in one go by least squares, like the line of step 8, instead of with a learning loop.
`np.linalg.lstsq` does that for a table with one column per ramp. **Guess first:** can the MSE ever
go up when you add a neuron?

In [ ]:
def fit_ramps(count):
    # Where the kinks go
    kinks = kink_positions(lowest, highest, count)
    # One column per neuron: its ramp for every building
    columns = []
    for kink in kinks:
        # The ramp with its kink here, for every building
        ramp = np.maximum(0, curve_x - kink)
        columns.append(ramp)
    # One more column, all ones (np.ones makes it): its weight is the
    # constant that lifts the whole curve
    ones = np.ones(len(curve_x))
    columns.append(ones)
    # np.column_stack puts the columns side by side into one table
    ramps = np.column_stack(columns)
    # Least squares: the weights with the lowest MSE (rcond=None just
    # picks the standard setting). It hands back several things; the
    # weights come first, hence [0].
    solution = np.linalg.lstsq(ramps, curve_y, rcond=None)
    weights = solution[0]
    # Weighted sum of the ramps for every building, and its MSE
    fitted = ramps @ weights
    error = np.mean((fitted - curve_y) ** 2)
    # Hand back two values: the kinks, and the weights with the error
    return kinks, [weights, error]


# The MSE for 1 to 16 neurons; errors[n - 1] belongs to n neurons
errors = []
for count in range(1, 17):
    # Catch the two values; fit[0] are the weights, fit[1] the MSE
    kinks, fit = fit_ramps(count)
    errors.append(fit[1])
    print(f"{count:2} neuron(s): MSE {fit[1]:7.2f}")

<details>
<summary>Answer</summary>

No. It goes from 379.1 with one neuron (exactly the best line of step 8) down to 0.04 with sixteen,
and now and then it barely moves (0.72 at both eight and nine). It cannot rise, because the kinks you
had are still there: the bigger model can do everything the smaller one could, and one thing more.
Had the kinks been moved around at every count, the MSE could get worse now and then.

</details>

Pull the slider from one neuron to sixteen. The lower panel shows each ramp already multiplied by its
weight, which is why some point downwards; the constant is not drawn.

In [ ]:
# Slider from 1 to 16 neurons: the sum on top, each weighted ramp below
grid = np.linspace(curve_x.min(), curve_x.max(), 400)
frames = []
steps = []
# Smallest and largest contribution, for the range of the lower panel
part_low = 0
part_high = 0
for count in range(1, 17):
    kinks, fit = fit_ramps(count)
    weights = fit[0]
    total = np.full(len(grid), weights[-1])
    lines = []
    for i in range(count):
        part = weights[i] * np.maximum(0, grid - kinks[i])
        total = total + part
        part_low = min(part_low, part.min())
        part_high = max(part_high, part.max())
        lines.append(go.Scatter(x=grid, y=part, mode="lines"))
    # Empty lines for the neurons not used yet (16 lines in every frame)
    for i in range(16 - count):
        lines.append(go.Scatter(x=grid, y=np.full(len(grid), np.nan)))
    data = [go.Scatter(x=grid, y=total, mode="lines")] + lines
    title = f"{count} neuron(s), MSE {fit[1]:.2f}"
    frames.append(go.Frame(name=str(count), data=data,
                           traces=list(range(1, 18)),
                           layout=go.Layout(title=title)))
    steps.append(dict(method="animate", label=str(count),
                      args=[[str(count)],
                            dict(mode="immediate",
                                 frame=dict(duration=0, redraw=False))]))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.6, 0.4],
                    subplot_titles=("the sum of the ramps, plus a constant",
                                    "what each neuron contributes"))
fig.add_trace(go.Scatter(x=curve_x, y=curve_y, mode="markers",
                         marker=dict(size=4, color="#4c78a8")), row=1, col=1)
first_frame = frames[0].data
fig.add_trace(go.Scatter(x=grid, y=first_frame[0].y, mode="lines",
                         line=dict(width=4, color="crimson")), row=1, col=1)
for i in range(16):
    fig.add_trace(go.Scatter(x=grid, y=first_frame[i + 1].y, mode="lines",
                             line=dict(width=1.5, color="#d8a24a")),
                  row=2, col=1)
fig.frames = frames
fig.update_layout(height=680, showlegend=False,
                  title=f"1 neuron(s), MSE {errors[0]:.2f}",
                  sliders=[dict(active=0,
                                currentvalue=dict(prefix="neurons: "),
                                steps=steps)])
fig.update_yaxes(range=[part_low - 20, part_high + 20], row=2, col=1)
fig.update_xaxes(title_text="compactness", row=2, col=1)
fig.show()

Sixteen kinks follow the curve closely. Hold that thought until next week, when we ask what happens
if you keep going until the model passes through every single point.

---

## Optional A: Part 1 on all 1073 buildings

The same two runs as in steps 4 and 6, on the whole table, written with the numpy shortcuts. The
gradients are still estimated with a tiny step.

In [ ]:
# Two columns of the table: df[list of names] keeps these columns,
# .to_numpy() makes them a numpy table
two_columns = df[["compactness", "u_value"]]
X = two_columns.to_numpy()
y = df["heating_demand"].to_numpy()
# .shape: (rows, columns)
print("table:", X.shape)


def mse_np(w1, w2, b, inputs):
    # The two weights as an array
    w = np.array([w1, w2])
    # Weighted sum for every row ...
    p = inputs @ w + b
    # ... then element by element, then the mean
    return np.mean((p - y) ** 2)


def learn_np(inputs, learning_rate, steps):
    # Blank start, as in step 4
    w1 = 0
    w2 = 0
    b = np.mean(y)
    tiny = 0.001
    # The MSE after every step, for the picture below
    history = []
    for step in range(steps):
        # The three gradients with a tiny step, as in step 3
        here = mse_np(w1, w2, b, inputs)
        further_w1 = mse_np(w1 + tiny, w2, b, inputs)
        further_w2 = mse_np(w1, w2 + tiny, b, inputs)
        further_b = mse_np(w1, w2, b + tiny, inputs)
        g_w1 = (further_w1 - here) / tiny
        g_w2 = (further_w2 - here) / tiny
        g_b = (further_b - here) / tiny
        # The three updates
        w1 = w1 - learning_rate * g_w1
        w2 = w2 - learning_rate * g_w2
        b = b - learning_rate * g_b
        # Note the MSE after this step
        now = mse_np(w1, w2, b, inputs)
        history.append(now)
    # Hand back two values: the three numbers, and the history
    return [w1, w2, b], history


# Raw inputs, 3000 steps. [-1] is the last element of a list.
raw_all, raw_history = learn_np(X, 0.015, 3000)
print(f"raw, 3000 steps: MSE {raw_history[-1]:.2f}")

# Rescaled: X.mean(axis=0) is a row of two averages; X minus that row
# subtracts each column's average from every row
averages = X.mean(axis=0)
spreads = X.std(axis=0)
X_scaled = (X - averages) / spreads
scaled_all, scaled_history = learn_np(X_scaled, 0.015, 400)
print(f"rescaled, 400 steps: MSE {scaled_history[-1]:.5f}")

# Back to real units, as in step 6, both weights at once
w_scaled = np.array([scaled_all[0], scaled_all[1]])
w_real = w_scaled / spreads
# np.sum adds up all elements: the two shifts of step 6 together
shifts = w_scaled * averages / spreads
b_real_all = scaled_all[2] - np.sum(shifts)
print(f"per unit of compactness {w_real[0]:.3f}, per W/m²K {w_real[1]:.3f}, "
      f"bias {b_real_all:.3f}")

Worse than on four buildings: after 3000 steps the raw run still sits at an MSE of 43.37. The
rescaled run is at 0.00002 after 400 steps and gives back 9.000, 61.993 and 18.012. The two runs as
curves, and the plane the model is, with the buildings sitting on it (turn it with the mouse).

In [ ]:
# Error per step of both runs, both axes logarithmic
fig = go.Figure()
fig.add_trace(go.Scatter(y=raw_history, mode="lines",
                         name="raw, 3000 steps"))
fig.add_trace(go.Scatter(y=scaled_history, mode="lines",
                         name="rescaled, 400 steps"))
fig.update_layout(height=400, xaxis_title="step", yaxis_title="MSE",
                  yaxis_type="log", xaxis_type="log",
                  title="what rescaling buys you")
fig.show()

In [ ]:
# The model as a plane over compactness and U-value, plus the buildings
c_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 30)
u_grid = np.linspace(X[:, 1].min(), X[:, 1].max(), 30)
C, U = np.meshgrid(c_grid, u_grid)
Z = w_real[0] * C + w_real[1] * U + b_real_all

fig = go.Figure()
fig.add_trace(go.Surface(x=c_grid, y=u_grid, z=Z, colorscale="Viridis",
                         opacity=0.55, showscale=False))
fig.add_trace(go.Scatter3d(x=X[:, 0], y=X[:, 1], z=y, mode="markers",
                           marker=dict(size=2, color=df["start_year"],
                                       colorscale="Turbo"),
                           name="buildings"))
fig.update_layout(height=600,
                  title="the model is a plane, and the buildings sit on it",
                  scene=dict(xaxis_title="compactness",
                             yaxis_title="U-value",
                             zaxis_title="heating demand"))
fig.show()

## Optional B: the matrix form

With sixteen neurons nobody writes sixteen lines. Each neuron has a weight on the input (here 1) and
a bias (minus its kink); the output has one weight per neuron and a constant. Collected in tables:
`W1` with one row per neuron, `b1`, `W2` and `b2`. Computing the output from the input is called the
**forward pass**. The cell does it once with loops and once with `@`, and compares.

In [ ]:
# The sixteen-neuron fit of step 11
kinks, fit = fit_ramps(16)
weights = fit[0]

# The tables. np.ones((16, 1)): 16 rows, 1 column, all ones.
W1 = np.ones((16, 1))
# Element by element: minus every kink
b1 = -np.array(kinks)
# The first 16 weights ([:16]), as a table of 16 rows and 1 column
W2 = weights[:16].reshape(16, 1)
# The last weight is the constant
b2 = weights[16]

# By foot: for every building, every neuron, then the weighted sum
by_foot = []
for x in curve_x:
    total = b2
    for i in range(16):
        # W1[i, 0]: row i, column 0 of the table
        z = W1[i, 0] * x + b1[i]
        # The ramp of neuron i, times its output weight
        total = total + relu(z) * W2[i, 0]
    by_foot.append(total)

# The same with tables. reshape(-1, 1): the inputs as one column.
x_column = curve_x.reshape(-1, 1)
# .T turns W1 on its side (1 row, 16 columns): one z per neuron and row
Z = x_column @ W1.T + b1
# All ramps at once
H = np.maximum(0, Z)
# Weighted sum of the 16 ramps, for every building
with_tables = H @ W2 + b2

# [:, 0]: all rows, column 0, so the result is a flat array again
flat = with_tables[:, 0]
# The largest difference between the two, over all buildings:
# np.abs drops the minus sign, .max() keeps the largest
gaps = np.abs(np.array(by_foot) - flat)
difference = gaps.max()
print(f"largest difference loop vs. tables: {difference:.2e}")
print("W1", W1.shape, " b1", b1.shape, " W2", W2.shape)

`W1` is called a **weight matrix**: one row per neuron, so a whole layer is one multiplication. The
difference is at the level of rounding (5.68e-14 means 0.0000000000000568): nothing new happens, the
same additions are written more compactly.

## Optional C: the same in PyTorch

From next week you write this yourself, so look at it once. Each line has a counterpart in what you
did by hand: the model is a layer of 16 ramps and an output, the loss is the MSE, and the
**optimiser**, the rule that turns gradients into steps, is gradient descent with a cleverer step
(Adam). The loop maps onto yours: `model(...)` is the prediction, `loss_function` the MSE,
`loss.backward()` computes all gradients at once (no tiny steps; PyTorch works them out exactly),
and `optimiser.step()` does all the updates. Unlike above, PyTorch also moves the kinks. The
homework uses the function `train_net` from this cell.

In [ ]:
import torch
import torch.nn as nn

# The data as PyTorch tensors (its arrays) ...
x_tensor = torch.tensor(curve_x, dtype=torch.float32)
y_tensor = torch.tensor(curve_y, dtype=torch.float32)
# ... each made one column: .unsqueeze(1) turns 1073 values into 1073
# rows of 1 value, the shape PyTorch layers expect
x_t = x_tensor.unsqueeze(1)
y_t = y_tensor.unsqueeze(1)
# Averages and spreads for rescaling input and target. torch's .std()
# divides by n - 1 instead of n; with 1073 buildings that hardly
# matters.
x_m = x_t.mean()
x_s = x_t.std()
y_m = y_t.mean()
y_s = y_t.std()
x_scaled = (x_t - x_m) / x_s
y_scaled = (y_t - y_m) / y_s
# The loss: the mean squared error
loss_function = nn.MSELoss()


def train_net(neurons, learning_rate, steps=2000, seed=0):
    # The same random start every time
    torch.manual_seed(seed)
    # A layer of `neurons` lines on one input, each with weight and bias
    layer = nn.Linear(1, neurons)
    # The kink behind each of them
    kink = nn.ReLU()
    # The output: a weighted sum of all neurons, plus a constant
    output = nn.Linear(neurons, 1)
    # The three pieces in a row
    model = nn.Sequential(layer, kink, output)
    # Adam, with all weights and biases of the model to move
    parameters = model.parameters()
    optimiser = torch.optim.Adam(parameters, lr=learning_rate)
    for step in range(steps):
        # Predict, and measure the loss
        prediction = model(x_scaled)
        loss = loss_function(prediction, y_scaled)
        # Forget old gradients, compute new ones, take one step
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    # Predict once more; no_grad: no gradients needed for this
    with torch.no_grad():
        scaled_out = model(x_scaled)
    # Undo the rescaling of the target
    out = scaled_out * y_s + y_m
    # As a flat numpy array: all rows, column 0
    out_table = out.numpy()
    out_np = out_table[:, 0]
    # The MSE against the target
    error = np.mean((out_np - curve_y) ** 2)
    # Hand back two values: the trained model and its MSE
    return model, error



# Sixteen neurons, learning rate 0.05, and the comparison
torch_model, torch_error = train_net(16, 0.05)
print(f"by hand, kinks by halving, weights solved: MSE {errors[15]:.2f}")
print(f"PyTorch, both layers trained, 2000 steps:  MSE {torch_error:.2f}")

The hand-built version wins by a wide margin. That is no argument against training, only about this
problem: one input, no noise, and an output that is a weighted sum, so its weights can be solved
exactly. Sixteen kinks by halving cover a line perfectly well. With two or more inputs, a fixed grid
of kinks would need far more neurons to cover the space as finely. Learning where the kinks go is
what makes networks work on anything wider than a line.

---

## Tests

First: Kernel, Restart Kernel and Run All Cells. Then this cell recomputes, on fresh numbers where it
can: your prediction for a known building, loop against numpy, the rescaling, the weights in real
units, two lines in a row, and the ramps.

In [ ]:
# assert checks something and stops with the message if it is false.
# 1e-9 means 0.000000001: "equal, up to tiny rounding".

# The rule itself: 9 * 6.263 + 62 * 1.5 + 18 = 167.367
known = predict(6.263, 1.5, 9, 62, 18)
assert abs(known - 167.367) < 1e-9, "predict() computes the wrong sum"

# Loop and numpy agree on another guess
loop_mse = mse(8, 50, 25, compactness, u_value)
test_table = np.array([[6.263, 1.50], [4.236, 1.45],
                       [5.195, 1.30], [5.175, 0.70]])
test_weights = np.array([8, 50])
test_p = test_table @ test_weights + 25
numpy_mse = np.mean((test_p - d4) ** 2)
assert abs(loop_mse - numpy_mse) < 1e-9, "loop and numpy disagree"

# Rescaling on fresh values: average 4, spread as numpy computes it
test_average, test_spread = average_and_spread([2.0, 4.0, 6.0])
assert abs(test_average - 4) < 1e-9, "the average is off"
numpy_spread = np.std([2.0, 4.0, 6.0])
assert abs(test_spread - numpy_spread) < 1e-9, "the spread is off"

# The weights in real units are the rule's: 9, 62 and 18
assert abs(w1_real - 9) < 0.05, f"w1 is {w1_real:.3f}, not 9"
assert abs(w2_real - 62) < 0.05, f"w2 is {w2_real:.3f}, not 62"
assert abs(b_real - 18) < 0.1, f"b is {b_real:.3f}, not 18"

# Two lines in a row (a = 2, p = 1, k = 3, q = -4): -1, 5, 11, 17
stacked = []
for x_test in [0, 1, 2, 3]:
    h_test = 2 * x_test + 1
    stacked.append(3 * h_test - 4)
assert stacked == [-1, 5, 11, 17], "two lines in a row are off"

# ReLU by foot equals np.maximum, and the two ramps give -10 at 8
for z in [-2.5, 0.0, 3.0]:
    assert relu(z) == np.maximum(0, z), "relu is off"
ramps_at_8 = 10 * relu(8 - 4) - 25 * relu(8 - 6)
assert ramps_at_8 == -10, "the two ramps are off"

# The MSE never rises with a neuron, and one neuron is the line
for n in range(1, 16):
    assert errors[n] <= errors[n - 1] + 1e-9, "MSE rose with a neuron"
assert abs(errors[0] - line_mse) < 1e-6, "one neuron should be the line"
print("✓ All green.")

## Where you stand now

You can put two inputs into one model and compute its prediction by hand. You know why the loop
crawled on raw inputs and walked straight in after rescaling, you can rescale a column yourself, and
you can read the weights back in units somebody can argue with. You have seen a straight model fail
on a curve and spotted the failure in the residuals rather than in the error number. And you can
build a curve out of kinks, which is the whole architectural idea behind every network in this
course. The numpy shortcuts you can now read: element by element, `@`, `np.mean` and `axis=0`.

What you should not take away is the collapse of the error in Part 1. That was arithmetic recovering
arithmetic.

Next session the target carries noise, you write PyTorch yourself, and the model that fits the
training data perfectly turns out to be the worse one.

### Terms to learn

These words are in the flashcard app, together with those of the other sessions, at
[gabriel-wurzer.github.io/data2/webapps/flashcards](https://gabriel-wurzer.github.io/data2/webapps/flashcards/).
Add it to your home screen; it works offline.

| term | in one sentence |
|---|---|
| normalisation | shifting each input to mean zero and spread one, so one learning rate suits all directions |
| standard deviation | the usual distance of a value from the mean of its column |
| feature | one input column; choosing them is a design decision, not a technicality |
| layer | several neurons computing side by side from the same inputs |
| ReLU | max(0, z); one kink, and the reason a network is more than a line |
| activation function | the non-linear step between two layers |
| weight matrix | one row per neuron, so a whole layer is a single multiplication (optional B) |
| forward pass | computing the output of the network from its inputs (optional B) |
| optimiser | the rule that turns gradients into steps; Adam is one (optional C) |
| residual pattern | structure left in the residuals, meaning the model lacks form |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

Before you pull updates of the course material, rename this notebook, for example by adding your
name, so your work is not overwritten.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

Take the PyTorch network from optional C above and fit it to the curved data. The collapsed cell
below defines it again (`train_net`), so the homework runs even if you skipped the optional parts. The first cell trains it with five different numbers of neurons,
at two learning rates that you choose. Everything else stays as it is, including the two thousand
steps and the starting seed.

1. Pick two learning rates between 0.001 and 0.2, the larger at least three times the smaller,
   not both from the examples in class, and enter them. Run the cell once. That gives you a table
   of ten errors; you do not have to search further.
2. The second cell draws your best network on a range wider than the data and prints what it
   says at two places outside. Keep the figure.

The table will not be tidy. That is part of the result, not a reason to leave numbers out.

In [ ]:
# The same as in optional C, so the homework also runs without it

import torch
import torch.nn as nn

# The data as PyTorch tensors (its arrays) ...
x_tensor = torch.tensor(curve_x, dtype=torch.float32)
y_tensor = torch.tensor(curve_y, dtype=torch.float32)
# ... each made one column: .unsqueeze(1) turns 1073 values into 1073
# rows of 1 value, the shape PyTorch layers expect
x_t = x_tensor.unsqueeze(1)
y_t = y_tensor.unsqueeze(1)
# Averages and spreads for rescaling input and target. torch's .std()
# divides by n - 1 instead of n; with 1073 buildings that hardly
# matters.
x_m = x_t.mean()
x_s = x_t.std()
y_m = y_t.mean()
y_s = y_t.std()
x_scaled = (x_t - x_m) / x_s
y_scaled = (y_t - y_m) / y_s
# The loss: the mean squared error
loss_function = nn.MSELoss()


def train_net(neurons, learning_rate, steps=2000, seed=0):
    # The same random start every time
    torch.manual_seed(seed)
    # A layer of `neurons` lines on one input, each with weight and bias
    layer = nn.Linear(1, neurons)
    # The kink behind each of them
    kink = nn.ReLU()
    # The output: a weighted sum of all neurons, plus a constant
    output = nn.Linear(neurons, 1)
    # The three pieces in a row
    model = nn.Sequential(layer, kink, output)
    # Adam, with all weights and biases of the model to move
    parameters = model.parameters()
    optimiser = torch.optim.Adam(parameters, lr=learning_rate)
    for step in range(steps):
        # Predict, and measure the loss
        prediction = model(x_scaled)
        loss = loss_function(prediction, y_scaled)
        # Forget old gradients, compute new ones, take one step
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    # Predict once more; no_grad: no gradients needed for this
    with torch.no_grad():
        scaled_out = model(x_scaled)
    # Undo the rescaling of the target
    out = scaled_out * y_s + y_m
    # As a flat numpy array: all rows, column 0
    out_table = out.numpy()
    out_np = out_table[:, 0]
    # The MSE against the target
    error = np.mean((out_np - curve_y) ** 2)
    # Hand back two values: the trained model and its MSE
    return model, error

In [ ]:
# The five network sizes to try
neuron_counts = [2, 4, 8, 16, 64]

# YOUR CHOICE: two learning rates between 0.001 and 0.2, the larger at
# least three times the smaller
learning_rates = [None, None]

# Check the choice before training
for rate in learning_rates:
    assert rate is not None, "enter two learning rates"
    assert 0.001 <= rate <= 0.2, "both rates between 0.001 and 0.2"
# min and max: the smaller and the larger of the two
smaller = min(learning_rates)
larger = max(learning_rates)
assert larger >= 3 * smaller, "the larger rate at least 3 times the smaller"

# One row per run: [neurons, learning rate, model, MSE]
results = []
for rate in learning_rates:
    for n in neuron_counts:
        # Train one network, keep it with its numbers, and show the error
        model, error = train_net(n, rate)
        results.append([n, rate, model, error])
        print(f"learning rate {rate:<7} {n:3d} neurons  error {error:9.3f}")
    # An empty line between the two learning rates
    print()
# The hand-built network of step 11, for comparison
print(f"by hand, 16 bends solved exactly: error {errors[15]:.3f}")
# How much smaller the error would be on the rescaled target
factor = float(y_s) ** 2
print("the error is a mean of squared misses, so its unit is the target's "
      "unit squared;")
print(f"on the scaled target it would have no unit and be {factor:.0f} "
      "times smaller")

In [ ]:
# The run with the lowest error: go through all rows, keep the best
best = results[0]
for row in results:
    if row[3] < best[3]:
        best = row
# Its trained model
best_model = best[2]


# The rule that made the data, for any compactness values
def rule(k):
    return 40 + 9 * k + 45 * np.sin(0.8 * k)


# What the best network says at the given compactness values
def net_at(values):
    # As a tensor, one column, as in optional C
    t = torch.tensor(values, dtype=torch.float32)
    column = t.unsqueeze(1)
    # Rescale, predict without gradients, undo the rescaling
    rescaled_in = (column - x_m) / x_s
    with torch.no_grad():
        scaled_out = best_model(rescaled_in)
    out = scaled_out * y_s + y_m
    # As a flat numpy array
    out_table = out.numpy()
    return out_table[:, 0]


# Draw network and rule from compactness 1 to 16, data shaded
wide = np.linspace(1.0, 16.0, 400)
wide_rule = rule(wide)
wide_net = net_at(wide)
fig = go.Figure()
fig.add_vrect(x0=curve_x.min(), x1=curve_x.max(), fillcolor="#4c78a8",
              opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=wide, y=wide_rule, mode="lines", name="the rule",
                         line=dict(width=2, color="#8b95a8", dash="dot")))
fig.add_trace(go.Scatter(x=curve_x, y=curve_y, mode="markers", name="data",
                         marker=dict(size=4, color="#4c78a8")))
fig.add_trace(go.Scatter(x=wide, y=wide_net, mode="lines",
                         name="your network",
                         line=dict(width=3, color="crimson")))
fig.update_layout(height=470, xaxis_title="compactness",
                  yaxis_title="target",
                  title=f"best run: {best[0]} neurons, learning rate "
                        f"{best[1]}, error {best[3]:.3f}; shaded where "
                        "there is data")
fig.show()

# The range of the data, and network and rule at two places outside
print(f"data from compactness {curve_x.min():.2f} to {curve_x.max():.2f}")
for k in [2.0, 15.0]:
    point = np.array([k])
    said = net_at(point)
    print(f"at compactness {k:4.1f}: network {said[0]:7.1f}, "
          f"rule {rule(k):7.1f}")

### Your interpretation

1. **Size.** Give your lowest error and its square root, the typical miss. Compare it with the sixteen bends placed by hand.
2. **More neurons.** At each of your two learning rates, did more neurons always give a lower error? Answer from your table, yes or no for each rate, with the numbers. Where the answer is no, name the step at which it breaks.
3. **Units.** The error is a mean of squared misses, computed after the scaling of the target was undone. In what units is it? The first cell prints by what factor it would shrink on the scaled target. The bends by hand were measured on the unscaled target; say why a number from the scaled target could not be compared with theirs.
4. **Range.** Give network and rule at the two printed points outside the data, and say how far your network departs from the rule on each side. The figure shows you what happens in between.
5. **One run that went wrong.** Your table has two runs for each number of neurons, one per learning rate. Find the pair whose errors differ most and give a possible reason. The table cannot prove it, and saying so is part of a good answer. If no pair differs much, say that, with the numbers, and what it suggests.
6. **Design consequence.** The first half of this session reached an error of nearly zero on the Vienna buildings. What does that number tell you about how well a heating-demand model would do on a real building? Answer it as if a client had asked.

*Your text here.*